In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

import commons, models, utils
from cough_datasets import CoughDatasets, CoughDatasetsCollate

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")
#######################################################################
parser = argparse.ArgumentParser()
parser.add_argument("--init", action="store_true")
parser.add_argument("--model_name", type=str, default="try_wavlmlora_downstream")
parser.add_argument("--config_path", type=str, default="configs/ssl_finetuning.json")
args = parser.parse_args(["--init", "--model_name", "try_opensmile", "--config_path", "configs/generalv2.json"])

model_dir = os.path.join("./logs", args.model_name)
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

config_save_path = os.path.join(model_dir, "config.json")
if args.init:
    with open(args.config_path, "r") as f:
        data = f.read()
    with open(config_save_path, "w") as f:
        f.write(data)
else:
    with open(config_save_path, "r") as f:
        data = f.read()
config = json.loads(data)

hps = utils.HParams(**config)
hps.model_dir = model_dir
hps.data.mae_training = hps.train.mae_training

# =============================================================
# SECTION: Loading Data
# =============================================================
df_train = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.train')
df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
example_data = df_train.sample(n=1)[['path_file']].values[0][0]

In [ ]:
import opensmile
import pandas as pd
import numpy as np
import soundfile as sf
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import joblib

In [ ]:
import opensmile
import pandas as pd
from pathlib import Path

FEATURE_SETS = [
    opensmile.FeatureSet.ComParE_2016,
    opensmile.FeatureSet.GeMAPSv01b,
    opensmile.FeatureSet.eGeMAPSv02,
    opensmile.FeatureSet.emobase,
    opensmile.FeatureSet.IS09,
    opensmile.FeatureSet.IS10,
    opensmile.FeatureSet.IS11,
    opensmile.FeatureSet.IS12,
    opensmile.FeatureSet.IS13,
]

SMILE_CLIENTS = {
    str(fs): opensmile.Smile(
        feature_set=fs,
        feature_level=opensmile.FeatureLevel.LowLevelDescriptors
    )
    for fs in FEATURE_SETS
}

def load_and_normalize(wav_path):
    audio, sr = sf.read(wav_path)
    audio = (audio - wav_mean) / (wav_std + 1e-6)
    return audio, sr

def extract_all_features(audio):
    outputs = []

    for fs_name, client in SMILE_CLIENTS.items():
        df = client.process_signal(audio, sr)
        df = df.reset_index(drop=True)
        df = df.add_prefix(fs_name + "__")
        outputs.append(df)

    merged = pd.concat(outputs, axis=1)
    merged = merged.fillna(0.0)
    merged["filepath"] = wav_path

    return merged

In [ ]:
import pickle as pk
from tqdm import tqdm

utils.compute_wav_stats(df_train, "path_file", pickle_path=f"precomputed_stats/opensmile_wavstats.pickle")
with open(f"precomputed_stats/opensmile_wavstats.pickle", 'rb') as f:
    stats = pk.load(f)
    wav_mean, wav_std = stats["mean_db"], stats["std_db"]

In [ ]:
all_dfs = []
for wav in tqdm(df_train['path_file'].tolist()):
    f = extract_all_features(wav)
    f = f.fillna(0.0)
    all_dfs.append(f)

In [ ]:
stacked = pd.concat(
    [df.drop(columns=["filepath"]) for df in all_dfs],
    axis=0,
    ignore_index=True
)

scaler = StandardScaler()
scaler.fit(stacked.values)

In [ ]:
import joblib
joblib.dump(scaler, "precomputed_stats/opensmile_global_scaler.pkl")
scaler = joblib.load("precomputed_stats/opensmile_global_scaler.pkl")

In [ ]:
f = extract_all_features(wav)
f = f.fillna(0.0)
f = f.drop(columns=["filepath"]).values

In [ ]:
f.shape

In [ ]:
scaler.transform(f)


In [ ]:
stacked.values.shape

In [ ]:
def fit_global_scaler(all_feature_frames):
    stacked = pd.concat(
        [df.drop(columns=["filepath"]) for df in all_feature_frames],
        axis=0,
        ignore_index=True
    )

    scaler = StandardScaler()
    scaler.fit(stacked.values)
    return scaler

In [ ]:
feats = extract_all_features(str(example_data))

In [ ]:
feats.shape

In [ ]:
feats.columns